# Lighthouse Vision

Run the cells in order in Google Colab with a T4 GPU and upload `lighthouse_dataset.zip`.
The notebook classifies all 180 screenshots and saves `preds.csv`, raw replies, and metrics.
This repeatedly evaluated cohort is descriptive, not an independent test.

Labels in saved results use spaces: `social proof` and `hidden information`.
The model prompt and JSON labels retain their original spelling internally to preserve inference behavior.
The model revision, quantization, image resolution, decoding, and retry settings match the supplied notebook.
GPU inference must be rerun to verify prediction equivalence; this notebook contains no precomputed predictions.

In [ ]:
%pip install -q transformers==5.19.0 accelerate==1.15.0 bitsandbytes==0.50.2 lm-format-enforcer==0.11.3 pandas==2.2.3 scikit-learn==1.6.1 pillow==11.3.0

In [ ]:
import gc
import json
import re
import shutil
import zipfile
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import classification_report, f1_score

DATA_DIR = Path('/content/lighthouse_data')
OUTPUT_DIR = Path('/content/lighthouse_results')
OUTPUT_DIR.mkdir(exist_ok=True)
archive = Path('/content/lighthouse_dataset.zip')
if not archive.exists():
    from google.colab import files
    files.upload()
with zipfile.ZipFile(archive) as dataset:
    dataset.extractall(DATA_DIR)

data = pd.read_csv(DATA_DIR / 'cohort_ocr.csv', keep_default_na=False)
for column in ['image', 'collector', 'label']:
    data[column] = data[column].astype(str).str.strip()
assert len(data) == 180 and data.image.is_unique

def image_identity(name):
    path = Path(name)
    return (path.stem if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'} else path.name).casefold()

def find_image(row):
    files = [path for path in (DATA_DIR / row.collector).iterdir() if path.is_file()]
    matches = [path for path in files if path.name == row.image]
    if not matches:
        matches = [path for path in files if image_identity(path.name) == image_identity(row.image)]
    if len(matches) != 1:
        raise ValueError(f'Missing or ambiguous screenshot: {row.collector}/{row.image}')
    with Image.open(matches[0]) as image:
        image.verify()
    return matches[0]

data['image_path'] = [find_image(row) for row in data.itertuples()]
print(f'Ready: {len(data)} screenshots')

In [ ]:
PROMPT = 'Classify one screenshot using only visible evidence. Do not speculate about content below the viewport, later interactions, factual truth of offers, or hidden behavior.\nEvaluate each category using these evidence tests:\nconfirmshaming: The decline button/link uses guilt, shame, negative identity or a loss-framed first-person refusal to pressure acceptance. This includes framing refusal as preferring to lose a benefit, spend unnecessarily, or lack a desirable identity. Inspect even small or low-contrast refusal text in offer/signup overlays. A bare decline is not enough: identify the evaluative or loss-loaded wording. Neutral Skip, No thanks, or Continue Without Offer does not qualify.\nhidden_information: A material renewal charge, mandatory fee, contractual condition or meaningful alternative is visibly difficult to notice because it is tiny, low contrast or suppressed. In pricing tables, a materially higher renewal price or recurring charge printed much smaller or lower contrast than a large introductory price qualifies as hidden_information even if the text is readable. Compare the visual prominence of the introductory price and the future charge. Normal scrolling, navigation, descriptions, decorative labels or a high price alone are not hidden information.\npreselection: An OPTIONAL seller-favoring choice is visibly already checked, toggled on, or selected by default, so the user must undo it. This includes paid add-ons, selected subscriptions, marketing consent and non-essential tracking. A visible checked box or enabled toggle establishes its current selected state; the choice need not be paid. Interpret control polarity: if a control disables, blocks or opts out of an optional seller-benefiting activity, its OFF state can leave that underlying activity enabled and therefore qualify as preselection. An OFF opt-out control is different from an OFF optional add-on. Necessary security or system settings do not qualify. A recommendation, Popular badge, colored card, upsell popup or plus button does not establish a selected choice.\nscarcity: Explicit remaining-stock or limited-quantity language such as only one left, almost sold out or people holding stock in carts. A discount, timer, review count, purchase count, bestseller badge or high price is not scarcity.\nurgency: An explicit countdown or deadline for an offer/action. Delivery-duration estimates are not countdowns. Limited time is urgency, not limited quantity.\nsocial_proof: Visible purchase counts or other people\'s purchase/viewing/activity claims are used as persuasion. Ordinary star ratings/reviews alone do not qualify. Sold counts can qualify.\nnone: All six evidence tests are negative.\nIf several categories have qualifying evidence, choose primary in this fixed order: confirmshaming, hidden_information, preselection, scarcity, urgency, social_proof. Put the other evidenced dark labels in others. Never invent a label to fill the list.\nReturn only valid JSON with EXACTLY three keys. primary is one of the seven exact label strings above. others is an array of distinct dark label strings excluding primary. evidence is ONE short string describing the actual visible cue, never an array or object. For none use others=[] and evidence="No qualifying visible cue."'

LABELS = {'confirmshaming', 'hidden_information', 'preselection', 'scarcity', 'urgency', 'social_proof', 'none'}
DARK_LABELS = LABELS - {'none'}

def readable_label(label):
    return label.replace('_', ' ')

def parse_reply(reply):
    try:
        match = re.search(r'\{.*\}', reply, re.DOTALL)
        result = json.loads(match.group())
        primary, others, evidence = result['primary'], result['others'], result['evidence']
        assert set(result) == {'primary', 'others', 'evidence'}
        assert isinstance(primary, str) and primary in LABELS
        assert isinstance(others, list) and all(isinstance(label, str) and label in DARK_LABELS for label in others)
        assert isinstance(evidence, str)
        assert len(others) == len(set(others)) and primary not in others
        assert primary != 'none' or not others
        return primary, others, evidence
    except (AttributeError, KeyError, TypeError, ValueError, AssertionError):
        return 'parse_fail', 'parse_fail', 'parse_fail'

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from transformers import PreTrainedTokenizerBase
import transformers.tokenization_utils as tokenizer_compat

# Compatibility required by the pinned format-enforcer version.
tokenizer_compat.PreTrainedTokenizerBase = PreTrainedTokenizerBase
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import (
    build_token_enforcer_tokenizer_data,
    build_transformers_prefix_allowed_tokens_fn,
)

MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
REVISION = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'
assert torch.cuda.is_available(), 'Select a GPU runtime.'

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=REVISION, quantization_config=quantization,
    device_map='auto', dtype=torch.float16,
)
model.eval()
processor = AutoProcessor.from_pretrained(
    MODEL_ID, revision=REVISION, min_pixels=256 * 28 * 28, max_pixels=2048 * 28 * 28,
)
patch_factor = processor.image_processor.patch_size * processor.image_processor.merge_size
processor.image_processor.size = {
    'shortest_edge': 256 * patch_factor ** 2,
    'longest_edge': 2048 * patch_factor ** 2,
}
tokenizer_data = build_token_enforcer_tokenizer_data(processor.tokenizer)
schema = {
    'type': 'object',
    'properties': {
        'primary': {'type': 'string', 'enum': sorted(LABELS)},
        'others': {'type': 'array', 'items': {'type': 'string', 'enum': sorted(DARK_LABELS)}, 'maxItems': 6},
        'evidence': {'type': 'string', 'maxLength': 180},
    },
    'required': ['primary', 'others', 'evidence'],
    'additionalProperties': False,
}

In [ ]:
def classify(path):
    with Image.open(path) as source:
        image = source.convert('RGB')
    instruction = PROMPT
    replies = []
    for attempt in range(2):
        messages = [{'role': 'user', 'content': [
            {'type': 'image'}, {'type': 'text', 'text': instruction},
        ]}]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
        allowed_tokens = build_transformers_prefix_allowed_tokens_fn(tokenizer_data, JsonSchemaParser(schema))
        with torch.inference_mode():
            generated = model.generate(
                **inputs, max_new_tokens=200 * (attempt + 1), do_sample=False,
                repetition_penalty=1.0, prefix_allowed_tokens_fn=allowed_tokens,
            )
        reply = processor.batch_decode(
            generated[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True,
        )[0].strip()
        del inputs, generated
        replies.append(reply)
        primary, others, evidence = parse_reply(reply)
        if primary in LABELS:
            break
        instruction += '\nReturn a complete JSON object with primary, others, evidence only.'
    return primary, others, evidence, replies

rows = []
for row in data.itertuples(index=False):
    gc.collect()
    torch.cuda.empty_cache()
    primary, others, evidence, replies = classify(row.image_path)
    rows.append({
        'image': row.image,
        'label': readable_label(row.label),
        'predicted': readable_label(primary),
        'others': json.dumps([readable_label(label) for label in others] if isinstance(others, list) else readable_label(others)),
        'evidence': evidence,
        'raw_reply': replies[-1],
        'raw_attempts': json.dumps(replies),
    })
    results = pd.DataFrame(rows)
    results.to_csv(OUTPUT_DIR / 'debug.csv', index=False)
    results[['image', 'label', 'predicted']].to_csv(OUTPUT_DIR / 'preds.csv', index=False)
    print(f'{len(rows)}/{len(data)}: {row.image} -> {readable_label(primary)}')

In [ ]:
predictions = pd.read_csv(OUTPUT_DIR / 'preds.csv', keep_default_na=False)
labels = sorted(readable_label(label) for label in LABELS)
valid = predictions.predicted.isin(labels)
correct = predictions.label.eq(predictions.predicted)
dark = predictions.label.ne('none')
metrics = {
    'n': len(predictions),
    'correct': int(correct.sum()),
    'accuracy': float(correct.mean()),
    'macro_f1': float(f1_score(predictions.label, predictions.predicted, labels=labels, average='macro', zero_division=0)),
    'dark_recall': float((dark & valid & predictions.predicted.ne('none')).sum() / dark.sum()) if dark.any() else None,
    'honest_false_alarms': int((~dark & valid & predictions.predicted.ne('none')).sum()),
    'honest_n': int((~dark).sum()),
    'parse_fails': int((~valid).sum()),
}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print(classification_report(predictions.label, predictions.predicted, labels=labels, zero_division=0))
archive = shutil.make_archive('/content/Lighthouse_Full_Results', 'zip', OUTPUT_DIR)
from google.colab import files
files.download(archive)